In [1]:
# mike babb
# created: 2026 08 23
# updated: 2026 09 23
# find five words with 25 different letters
# example: ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']

In [2]:
# standard
import math
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations, combinations

In [3]:
import pandas as pd
import numpy as np

In [4]:
# custom
from utils import *
import _run_constants as rc

# LOAD DATA

In [5]:
word_df, word_id_list, word_byte_list, word_byte_array, word_byte_to_word_dict = load_input_data()

In [6]:
# confirm that these words - a valid solution - are present
outcome_words = ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']
for ow in outcome_words:
    print(ow, ''.join(sorted(ow)), ''.join(sorted(ow)) in word_df['letters_sorted'].tolist())

vibex beivx True
glyph ghlpy True
muntz mntuz True
dwarf adfrw True
jocks cjkos True


## DEMONSTRATE BITWISE OPERATIONS

In [7]:
vibex = byte_encode_words('vibex')
glyph = byte_encode_words('glyph')
muntz = byte_encode_words('muntz')
dwarf = byte_encode_words('dwarf')
jocks = byte_encode_words('jocks')
cramp = byte_encode_words('cramp')

In [8]:
# this is equal to zero - no letters reused
(vibex | glyph | muntz | dwarf) & jocks

0

In [9]:
# this is not equal to zero because letters are reused
((vibex | glyph) | muntz | dwarf) & cramp

167937

In [10]:
# order of operations for bitwise operations
(vibex | glyph) & (muntz | dwarf) 

0

In [11]:
vibex | glyph | muntz | dwarf | jocks

67043327

In [12]:
# same output as above
byte_encode_words('vibexglyphmuntzdwarfjocks')

67043327

# BUILD LEVEL 2 BY COMBINING TWO BYTE ENCODED WORDS

In [13]:
l2_list = np.full(shape = (10000000, 3), fill_value = -1, dtype = np.int32)
row_index = 0
found_values = set()
for w1_be, w2_be in combinations(word_byte_list, 2):
    if w1_be & w2_be == 0:   
        # they share no letters in common, compute the bitwise or to add the words together
        l2 = w1_be | w2_be                          
        
        l2_list[row_index, :] = np.array([w1_be, w2_be, l2], dtype = np.int32)
        found_values.add(l2)
        row_index += 1

# trim the data frame
l2_list = l2_list[:row_index, :]
l2_df = pd.DataFrame(data = l2_list, columns = ['w1b', 'w2b', 'l2'])
l2_df.shape

(3213696, 3)

# BUILD LEVELS 4 AND 5 BY COMBINING TWO ITEMS FROM THE L2 LIST  
# COMPARE THAT WITH THE WORD BYTE ARRAY ONE MORE TIME

In [14]:
# get words from bytes
l2_df['w1'] = l2_df['w1b'].map(word_byte_to_word_dict)
l2_df['w2'] = l2_df['w2b'].map(word_byte_to_word_dict)

In [15]:
w_l2_df = l2_df.drop_duplicates(subset = 'l2').reset_index(drop = True)

In [16]:
l2_all = w_l2_df['l2'].to_numpy(dtype = np.int32)

In [17]:
# let's just use the word jocks
w_l2_df = w_l2_df.loc[(w_l2_df['w1'] == 'jocks') |
                              (w_l2_df['w2'] == 'jocks'), ['w1b', 'w2b', 'l2']].reset_index(drop = True)

In [18]:
w_l2_df.shape

(928, 3)

In [19]:
w_l2_df['l2'].unique().shape

(928,)

In [20]:
# we are going to make a lot of comparisons
print('The full set of l2 - duplicated l2:', l2_df.shape[0], l2_df.shape[0] ** 2)
print('The unique l2:', l2_df['l2'].unique().shape[0],  l2_df['l2'].unique().shape[0]** 2)
# but, we'll be clever about this and compare each item from the l2_list
# against the whole l2_list using array operations. 


The full set of l2 - duplicated l2: 3213696 10327841980416
The unique l2: 640023 409629440529


# COMPUTE THE COMBINATIONS

In [21]:
l2_all.shape

(640023,)

In [22]:
w_l2_df.shape

(928, 3)

In [ ]:
# compute all possible pairs
start_pos = 0
total_output = np.zeros(shape = (100_000_000, 5), dtype = np.int32)
for i_row, row in w_l2_df.iterrows():    
    # levels 1 and 2
    w1b, w2b, l2 = row

    # compare the current l2 to all l2 - this will find all instances
    # a value of zero indicates that there are no letters in common
    # indexer for l2, l3, and l4
    # bitwise and to identify two l2 that do not have a letter in common
    positional_idx_l2l3l4 = (l2_all & l2) == 0 
    # TODO: fix this!
    if positional_idx_l2l3l4.any():

        # these are l2 words with different letters the the l2 above. 
        # this is effectively l4
        # this is now 20 different letters
        output_array_w3bw4b = l2_all[positional_idx_l2l3l4]    
        
        # l2, l3, l4 accumulated letters
        output_array_l2l3l4 = output_array_w3bw4b | l2

        # create the temp output
        n_rows_l2l3l4 = output_array_l2l3l4.shape[0]
        print(n_rows_l2l3l4)

        # check against the word_byte_array for w5b        
        for w3bw4b, l2l3l4 in zip(output_array_w3bw4b, output_array_l2l3l4):
            # w3bw4b: this is the other l2. l2 and w3bw4b have no letters in common. 20 letters.

            # create an indexer
            positional_idx_l2l3l4l5 = (word_byte_array & w3bw4b) == 0

            if positional_idx_l2l3l4l5.any(): 
                # the indexer has at least one True value.
                
                # output_array_l2l3l4l5 is the list of word(s) that have letters
                # that do not match the other letters. In other words, this is 
                # the final five letters not in the group of twenty
                output_array_l2l3l4l5 = word_byte_array[positional_idx_l2l3l4l5]

                # count!
                n_rows_l2l3l4l5 = output_array_l2l3l4l5.shape[0]

                # create a temporary matrix to hold the output
                temp_output = np.zeros(shape = (n_rows_l2l3l4l5, 5), dtype = np.int32)
                # word 1
                temp_output[:, 0] = w1b
                # word 2
                temp_output[:, 1] = w2b
                # the bitwise or on w1 and w2
                temp_output[:, 2] = l2

                # calculate w3b and w4b
                # this is the 'other' w1b and w2b values - w3b and w4b, effectively.
                temp_output[:, 3] = w3bw4b
                # the final word
                temp_output[:, 4] = output_array_l2l3l4l5

                # update the total output with the temporary list
                total_output[start_pos:start_pos + n_rows_l2l3l4l5, :] = temp_output

                # the counter
                start_pos += n_rows_l2l3l4l5
                print(start_pos) 
    
    if i_row % 1000 == 0:
        # print the number of l2 iterations and the shape of the output
        print(i_row)   

68
344
569
890
1136
1377
1656
1879
2150
2396
2669
2990
3291
3577
3840
4144
4456
4851
5227
5623
5968
6298
6617
6985
7348
7606
8029
8333
8697
8944
9205
9647
10094
10442
10678
10963
11339
11723
12091
12476
12802
13007
13245
13559
13891
14136
14416
14700
14964
15205
15484
15795
16152
16443
16675
16949
17296
17549
17803
18023
18349
18548
18745
18965
19224
19440
19776
20120
20439
0
9
20658
21105
21529
21950
22307
22655
22873
23274
23561
7
23759
23982
24453
24938
25252
25633
26080
73
26401
26663
26904
27159
27527
27866
28228
28572
28995
29226
29504
29788
30026
30298
30648
31003
31324
31590
31894
32258
32571
32958
33292
33668
33982
34366
34775
35143
35511
35896
36228
36457
36724
37031
37295
37618
37922
38270
38512
38803
39204
39512
39859
40177
40415
40711
41069
41293
41547
41866
42137
42447
42759
42974
43274
43595
43901
44186
44431
44699
45037
45373
45654
45928
46207
46536
46831
47215
47559
47976
48295
48698
49126
39
49494
49781
50132
50483
50821
51163
51481
51779
52152
52435
52768
53135
53469

# CREATE AND SHAPE THE OUTPUT

In [ ]:
output = total_output[:start_pos]

In [ ]:
output.shape

In [ ]:
# turn it into a dataframe
output_df = pd.DataFrame(data = output, columns = ['w1b', 'w2b', 'l2',  'l3l4', 'w5b'])

In [ ]:
output_df.head()

# JOIN TO GET THE W3B AND THE W4B

In [ ]:
l3l4_df = l2_df[['w1b', 'w2b', 'l2']].copy()
l3l4_df.columns = ['w3b', 'w4b', 'l3l4',]

In [ ]:
l3l4_df.shape

In [ ]:
output_df = pd.merge(left = output_df, right = l3l4_df)

In [ ]:
output_df.shape

In [ ]:
output_df.head()

In [ ]:
# reorder...
col_names = ['w1b', 'w2b', 'w3b', 'w4b', 'w5b']
output_df = output_df[col_names].copy()

In [ ]:
# get words!
for ii in range(1, 6):
    bcn = f"w{ii}b"
    cn = f"w{ii}"
    output_df[cn] = output_df[bcn].map(word_byte_to_word_dict)

In [ ]:
# count the remainder letter
lc_set = set(ascii_lowercase)
def get_remainder_letter(row):
    my_set = set()
    for cn in ['w1', 'w2', 'w3', 'w4', 'w5']:
        my_set.update(row[cn])

    return ''.join(lc_set.difference(my_set))

output_df['remaining_letter'] = output_df.apply(get_remainder_letter, axis = 1)



In [ ]:
# count unique words - JUST TO VERIFY
col_names = ['w1', 'w2', 'w3', 'w4', 'w5']
output_df['n_unique_words'] = output_df[col_names].apply(lambda x: len(set(x)), axis = 1)

In [ ]:
# add the words - ALSO TO VERIFY
output_df['bitwise_or'] = 0
output_df['bitwise_and'] = 0
for cn_idx in range(1, 6):
    b_cn = f"w{cn_idx}b"
    w_cn = f"w{cn_idx}"
    output_df[w_cn] = output_df[b_cn].map(word_byte_to_word_dict)
    output_df['bitwise_and'] = output_df['bitwise_and'] & output_df[b_cn]
    output_df['bitwise_or'] = output_df['bitwise_or'] | output_df[b_cn]


In [ ]:
output_df.head()

# CREATE AND SAVE OUTPUT

In [ ]:
output_df.to_excel(excel_writer='test.xlsx', index = False)